# NeuralScene Bench

Controlled novel-view synthesis benchmark for Netflix JR40251 portfolio work.

Initial scope: Mip-NeRF 360 Bonsai, comparing Nerfstudio `splatfacto` and `nerfacto` under one shared Nerfstudio evaluation pipeline.

Pinned Nerfstudio source revision: `50e0e3c70c775e89333256213363badbf074f29d`.


## Step 1. Check the Colab GPU environment


In [ ]:
import sys, subprocess
print('Python:', sys.version)
import torch
print('PyTorch:', torch.__version__)
print('CUDA build:', torch.version.cuda)
print('CUDA available:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)
subprocess.run(['nvidia-smi'])


## Step 2. Install the pinned benchmark environment


In [ ]:
from pathlib import Path
import subprocess, sys
NS_COMMIT = '50e0e3c70c775e89333256213363badbf074f29d'
VENV = Path('/content/neuralscene-env')
NS = Path('/content/nerfstudio')
PY = VENV / 'bin/python'
def run(cmd, cwd=None):
    print('\n$', ' '.join(map(str, cmd)), flush=True)
    subprocess.run(list(map(str, cmd)), cwd=cwd, check=True)
run([sys.executable, '-m', 'pip', 'install', '-q', 'uv'])
if not PY.exists(): run(['uv', 'venv', '--python', '3.11', VENV])
run(['uv', 'pip', 'install', '--python', PY, 'torch==2.7.1', 'torchvision==0.22.1', '--index-url', 'https://download.pytorch.org/whl/cu128'])
if not NS.exists(): run(['git', 'clone', 'https://github.com/nerfstudio-project/nerfstudio.git', NS])
run(['git', 'fetch', '--all', '--tags'], cwd=NS)
run(['git', 'reset', '--hard', NS_COMMIT], cwd=NS)
run(['uv', 'pip', 'install', '--python', PY, '-e', NS])
run([PY, '-c', "import torch; print('Environment ready'); print('Torch:', torch.__version__); print('CUDA:', torch.version.cuda); print('CUDA available:', torch.cuda.is_available()); print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)"])
print('\nPinned Nerfstudio commit:')
run(['git', 'rev-parse', 'HEAD'], cwd=NS)


## Step 3. Prepare the actual Mip-NeRF 360 Bonsai dataset

Correction: the earlier CIIRC `gaussian-splatting/mipnerf360/bonsai.zip` URL is a benchmark-result artifact containing a trained checkpoint and predictions, not the source dataset. This cell instead reads Bonsai directly from the official Google Research Mip-NeRF 360 archive. It uses HTTP range access and extracts only `bonsai/images_2` and `bonsai/sparse`, so it does not intentionally materialize the full multi-scene archive on disk.


In [ ]:
from pathlib import Path
import subprocess, sys, shutil, zipfile

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'fsspec', 'aiohttp'], check=True)
import fsspec

URL = 'https://storage.googleapis.com/gresearch/refraw360/360_v2.zip'
ROOT = Path('/content/data/mipnerf360')
SCENE = ROOT / 'bonsai'
ROOT.mkdir(parents=True, exist_ok=True)

# Remove files from the incorrect benchmark-artifact attempt to reclaim disk space.
wrong_zip = Path('/content/bonsai_mipnerf360.zip')
wrong_tmp = ROOT / '_bonsai_extract'
if wrong_zip.exists():
    print('Removing previous benchmark-result ZIP:', wrong_zip)
    wrong_zip.unlink()
shutil.rmtree(wrong_tmp, ignore_errors=True)

ready = (SCENE/'images_2').is_dir() and (SCENE/'sparse/0').is_dir()
if not ready:
    shutil.rmtree(SCENE, ignore_errors=True)
    SCENE.mkdir(parents=True, exist_ok=True)
    print('Opening official Mip-NeRF 360 archive with HTTP range access...', flush=True)
    with fsspec.open(URL, 'rb', block_size=16 * 1024 * 1024, cache_type='readahead') as remote:
        with zipfile.ZipFile(remote) as zf:
            wanted = [
                info for info in zf.infolist()
                if info.filename.startswith('bonsai/images_2/')
                or info.filename.startswith('bonsai/sparse/')
            ]
            print('Selected archive entries:', len(wanted))
            if not wanted:
                raise RuntimeError('Bonsai images_2/sparse entries were not found in the official archive.')
            for i, info in enumerate(wanted, 1):
                rel = Path(info.filename).relative_to('bonsai')
                target = SCENE / rel
                if info.is_dir():
                    target.mkdir(parents=True, exist_ok=True)
                    continue
                target.parent.mkdir(parents=True, exist_ok=True)
                with zf.open(info) as src, open(target, 'wb') as dst:
                    shutil.copyfileobj(src, dst, length=8 * 1024 * 1024)
                if i % 25 == 0:
                    print(f'Extracted {i}/{len(wanted)} entries', flush=True)

images = sorted(p for p in (SCENE/'images_2').iterdir() if p.is_file()) if (SCENE/'images_2').is_dir() else []
colmap = SCENE/'sparse/0'
print('\nBonsai root:', SCENE)
print('images_2 exists:', (SCENE/'images_2').is_dir())
print('Image count:', len(images))
print('sparse/0 exists:', colmap.is_dir())
print('COLMAP files:', sorted(p.name for p in colmap.iterdir()) if colmap.is_dir() else [])
if not images or not colmap.is_dir():
    raise RuntimeError('Bonsai dataset is incomplete after extraction.')
print('\nBONSAI DATA READY')


## Next

After Step 3 ends with `BONSAI DATA READY`, add a short Splatfacto smoke test before running the full benchmark.
